# Information Retrieval to RAG

Build a complete information retrieval and RAG pipeline with Oracle AI Database and OpenAI.

![Information Retrieval](../images/IR.png)

## What You Will Build

A **Research Paper Assistant** — an AI-powered system that searches, retrieves, and reasons over 200 arxiv papers stored in Oracle AI Database. You will progress from basic retrieval to a full RAG pipeline that connects Oracle search to OpenAI generation.

## What You Will Learn

In this notebook you will learn how to:
- Set up Oracle AI Database locally for AI applications
- Load, prepare, and embed research paper data
- Implement 5 retrieval strategies (keyword, vector, hybrid pre/post filter, RRF, graph)
- Build an end-to-end RAG pipeline

## Select Your Kernel

Before running any code cells, select the correct Python kernel:

**Step 1.** Click **Select Kernel** in the top-right corner of the notebook.

![Select Kernel](../images/select_kernel.png)

**Step 2.** Choose **Python 3.11** from the list.

![Select Python 3.11](../images/select_kernel_python_3.11.png)

**Step 3.** Confirm the kernel shows as selected in the top-right corner.

![Kernel Selected](../images/ensure_kernel_selected.png)

> **Note:** If you don't see the kernel, the Codespace may still be setting up. Wait for the terminal to show `Setup complete!` and try again.

> **Note:** Dependencies are pre-installed in GitHub Codespaces. If running locally, uncomment the install cell below.

In [ ]:
# ! pip install -Uq oracledb pandas sentence-transformers datasets einops "numpy<2.0" openai

# Part 1: Oracle AI Database Setup

**Oracle AI Database** is a converged database built for AI developers. It unifies relational, document, graph, and vector data with native support for `VECTOR` columns, HNSW indexes, `VECTOR_DISTANCE()`, Oracle Text, and SQL Property Graphs. For this workshop we use a local Docker installation of Oracle AI Database Free.

> 💡 **Key Term — Converged Database:** Oracle AI Database unifies relational, vector, graph, and document data in a single engine. 
This means your embeddings, metadata, and graph relationships live side-by-side — no need to sync across separate systems.

## 1.1 Docker Setup

> If running in **GitHub Codespaces**, Oracle is already started for you — skip to the next cell.
>
> If running locally, see the [README](../README.md) for Docker setup instructions.

### 1.1.1 Connecting to Oracle AI Database

> 📖 **Companion guide:** See [docs/part-1-oracle-setup.md](../docs/part-1-oracle-setup.md) for detailed explanation and troubleshooting.

The cells below define a connection helper with retry logic and connect to Oracle. 
Property graph privileges are granted automatically during Codespace setup.

In [ ]:
import os
import oracledb
import time

`connect_to_oracle()` attempts to connect up to 3 times with a 5-second delay between retries. 
Oracle may still be initialising when the notebook starts — retry logic handles this gracefully. 
On success it prints the database version banner to confirm the connection.

In [ ]:
def connect_to_oracle(max_retries=3, retry_delay=5):
    """Connect to Oracle database with retry logic."""
    user = "VECTOR"
    password = "VectorPwd_2025"
    dsn = "localhost:1521/FREEPDB1"

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(user=user, password=password, dsn=dsn)
            print("Connected successfully!")
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%'")
                print(cur.fetchone()[0])
            return conn
        except oracledb.OperationalError as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                print(f"Retrying in {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                raise

In [ ]:
conn = connect_to_oracle()

> 💡 **Takeaway — One Database, Many Capabilities:** You just connected to Oracle AI Database — the same engine that will store your vectors, run keyword searches, execute graph queries, and serve as your RAG pipeline's retrieval backend. No external vector database, no separate search service. Everything lives in one place with ACID guarantees.

# Part 2: Data Loading, Preparation & Embedding Generation

> 📖 **Companion guide:** See [docs/part-2-data-loading.md](../docs/part-2-data-loading.md)

> 💡 **Key Term — Embedding:** A numeric vector (list of floats) that captures the semantic meaning of text. 
Similar meanings produce similar vectors, enabling search by meaning rather than keywords.

## 2.1 Data Loading From Hugging Face

![Data Loading](../images/data_loading.png)

We stream 200 ArXiv papers using `streaming=True` to avoid downloading the full dataset.

In [ ]:
import pandas as pd
from datasets import load_dataset

ds_stream = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

We iterate through the streamed dataset and extract five fields per paper:

- **arxiv_id** — unique identifier for deduplication and graph edges
- **title** and **abstract** — the text we will embed and search over
- **authors** — normalised to a list for the author-paper graph
- **primary_subject** — category label useful for filtering

A combined `text` field (title + abstract) is created for embedding, since both carry semantic signal.

> **Heads up:** The cell below downloads the dataset from Hugging Face. This typically takes around **1 minute** depending on your internet connection — sit tight while it streams in.

In [ ]:
sampled = []
for i, item in enumerate(ds_stream):
    if i >= 200:
        break

    arxiv_id = item.get("arxiv_id", f"unknown_{i}")
    title = item.get("title", "").strip()
    abstract = item.get("abstract", "").strip()
    authors = item.get("authors", [])

    if isinstance(authors, str):
        authors = [a.strip() for a in authors.split(",") if a.strip()]
    elif isinstance(authors, list):
        authors = [str(a).strip() for a in authors if str(a).strip()]
    else:
        authors = []

    text = f"{title} — {abstract}"

    sampled.append({
        "id": item.get("id", f"arxiv_{i}"),
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "authors": authors,
        "text": text
    })

print(f"Streamed {len(sampled)} papers.")

In [ ]:
dataset_df = pd.DataFrame(sampled)
print(f"Loaded {len(dataset_df)} rows into DataFrame.")
dataset_df.head()

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

![Embedding Generation](../images/embedding_generation.png)

## 2.2 Embedding Generation

The Nomic embedding model uses **task prefixes** to distinguish how text should be encoded. This is called **asymmetric embedding** — the idea that documents and queries play fundamentally different roles in retrieval, so they should be encoded differently.

- `search_document:` — prepended to text when **storing documents**. This tells the model to capture the full topical content of the passage, optimizing the embedding for being *found*.
- `search_query:` — prepended to text when **encoding a user query**. This tells the model to focus on the *intent* behind the question, optimizing the embedding for *finding* relevant documents.

**Why does this matter?** A user query like *"What are the latest advances in reinforcement learning?"* is structurally very different from a research paper abstract about reinforcement learning. Without prefixes, the model would encode both the same way, which can reduce retrieval quality. With prefixes, Nomic learns two complementary encoding strategies — one tuned for short, intent-driven queries and another for longer, content-rich documents — so the resulting vectors align better in the shared embedding space.

**Impact on retrieval:** In practice, asymmetric prefixing consistently improves recall and precision compared to encoding everything the same way. If you forget to add the `search_query:` prefix at query time (or vice versa), you may notice a drop in retrieval relevance because the embeddings won't be in the same "mode" the model expects.

> **Note:** Encoding all documents takes approximately **1–2 minutes** as each text is individually embedded. You will see a progress counter updating in the output below.

In [ ]:
import numpy as np
import sys

dataset_df["text_prefixed"] = dataset_df["text"].apply(lambda x: f"search_document: {x}")
texts = dataset_df["text_prefixed"].tolist()
embs = []

total = len(texts)
for i, text in enumerate(texts, start=1):
    embedding = embedding_model.encode(
        text, show_progress_bar=False,
        convert_to_numpy=True, normalize_embeddings=True
    )
    embs.append(embedding)
    sys.stdout.write(f"\rEncoding {i}/{total} texts...")
    sys.stdout.flush()

print(f"\nFinished encoding {len(embs)} texts.")
embs = np.vstack(embs)
print(f"Embeddings shape: {embs.shape}")

In [ ]:
dataset_df["embedding"] = [e.astype(np.float32).tolist() for e in embs]
dim = len(dataset_df["embedding"].iloc[0])
print(f"Embedding dimension: {dim}")
dataset_df.head(2)

> 💡 **Takeaway — Embeddings Are the Bridge:** You turned 200 research papers into 768-dimensional vectors using a local model — no API calls, no data leaving your machine. The asymmetric prefix scheme (`search_document:` vs `search_query:`) is easy to overlook but critical: it tells the model whether text is *content to be found* or a *question seeking content*. Forgetting the prefix at query time silently degrades results.

# Part 3: Database Table Setup & Data Ingestion

> 📖 **Companion guide:** See [docs/part-3-table-setup.md](../docs/part-3-table-setup.md)

> 💡 **Key Term — HNSW Index:** Hierarchical Navigable Small World — a graph-based algorithm for fast approximate nearest-neighbour search. 
Without it, every query scans all vectors. With it, queries run in milliseconds even at millions of rows.

## 3.1 Create Research Papers Table

In [ ]:
ddl = f"""
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE paper_similarities';
EXCEPTION WHEN OTHERS THEN
    IF SQLCODE != -942 THEN RAISE; END IF;
END;
/
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE paper_authors';
EXCEPTION WHEN OTHERS THEN
    IF SQLCODE != -942 THEN RAISE; END IF;
END;
/
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE authors';
EXCEPTION WHEN OTHERS THEN
    IF SQLCODE != -942 THEN RAISE; END IF;
END;
/
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE research_papers CASCADE CONSTRAINTS PURGE';
EXCEPTION WHEN OTHERS THEN
    IF SQLCODE != -942 THEN RAISE; END IF;
END;
/
CREATE TABLE research_papers (
    arxiv_id VARCHAR2(255) PRIMARY KEY,
    title VARCHAR2(4000),
    abstract VARCHAR2(4000),
    text CLOB,
    embedding VECTOR({dim}, FLOAT32)
)
TABLESPACE USERS
"""

In [ ]:
with conn.cursor() as cur:
    for stmt in ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)
conn.commit()
print(f"Table RESEARCH_PAPERS created with VECTOR dimension: {dim}")

## 3.2 Create Indexes (Vector and Search)

Vector indexes accelerate similarity search from a full table scan to sub-second lookups. Oracle supports two types:

| Index | How It Works | Best For |
|---|---|---|
| **HNSW** | Graph-based — builds a navigable small-world graph at insert time | Low-latency queries, moderate data sizes |
| **IVF** | Partition-based — clusters vectors into buckets, searches nearby buckets | Large datasets where build time matters less |

We create both below so you can compare. In production, HNSW is the most common choice for interactive applications.

In [ ]:
with conn.cursor() as cur:
    for idx_name in ("RP_VEC_HNSW", "RP_VEC_IVF"):
        try:
            cur.execute(f"DROP INDEX {idx_name}")
        except oracledb.DatabaseError as e:
            if "ORA-01418" not in str(e):
                raise

    cur.execute("""
        CREATE VECTOR INDEX RP_VEC_HNSW
        ON research_papers(embedding)
        ORGANIZATION INMEMORY NEIGHBOR GRAPH
        DISTANCE COSINE
        WITH TARGET ACCURACY 90
        PARAMETERS (TYPE HNSW, NEIGHBORS 40, EFCONSTRUCTION 500)
        TABLESPACE USERS
    """)
conn.commit()
print("Vector Index RP_VEC_HNSW (HNSW) created")

In [ ]:
with conn.cursor() as cur:
    try:
        cur.execute("DROP INDEX rp_text_idx")
    except oracledb.DatabaseError as e:
        if "ORA-01418" not in str(e):
            raise

    cur.execute("""
        CREATE INDEX rp_text_idx
        ON research_papers(text)
        INDEXTYPE IS CTXSYS.CONTEXT
        PARAMETERS ('SYNC (ON COMMIT)')
    """)
conn.commit()
print("Oracle Text index (rp_text_idx) created on TEXT column.")

### 3.2.1 Create Relational Tables for Graph Retrieval

In [ ]:
graph_tables_ddl = """
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE paper_similarities';
EXCEPTION WHEN OTHERS THEN
    IF SQLCODE != -942 THEN RAISE; END IF;
END;
/
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE paper_authors';
EXCEPTION WHEN OTHERS THEN
    IF SQLCODE != -942 THEN RAISE; END IF;
END;
/
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE authors';
EXCEPTION WHEN OTHERS THEN
    IF SQLCODE != -942 THEN RAISE; END IF;
END;
/
CREATE TABLE authors (
    author_name VARCHAR2(512) PRIMARY KEY
)
TABLESPACE USERS
/
CREATE TABLE paper_authors (
    arxiv_id VARCHAR2(255) NOT NULL,
    author_name VARCHAR2(512) NOT NULL,
    CONSTRAINT pk_paper_authors PRIMARY KEY (arxiv_id, author_name),
    CONSTRAINT fk_pa_paper FOREIGN KEY (arxiv_id) REFERENCES research_papers(arxiv_id),
    CONSTRAINT fk_pa_author FOREIGN KEY (author_name) REFERENCES authors(author_name)
)
TABLESPACE USERS
/
CREATE TABLE paper_similarities (
    source_arxiv_id VARCHAR2(255) NOT NULL,
    target_arxiv_id VARCHAR2(255) NOT NULL,
    sim_score NUMBER(8,6) NOT NULL,
    rank_no NUMBER(5) NOT NULL,
    CONSTRAINT pk_paper_similarities PRIMARY KEY (source_arxiv_id, target_arxiv_id),
    CONSTRAINT fk_ps_src FOREIGN KEY (source_arxiv_id) REFERENCES research_papers(arxiv_id),
    CONSTRAINT fk_ps_tgt FOREIGN KEY (target_arxiv_id) REFERENCES research_papers(arxiv_id),
    CONSTRAINT ck_ps_not_self CHECK (source_arxiv_id <> target_arxiv_id)
)
TABLESPACE USERS
"""

with conn.cursor() as cur:
    for stmt in graph_tables_ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)
    cur.execute("CREATE INDEX idx_pa_author ON paper_authors(author_name)")
    cur.execute("CREATE INDEX idx_ps_source ON paper_similarities(source_arxiv_id)")
    cur.execute("CREATE INDEX idx_ps_target ON paper_similarities(target_arxiv_id)")
conn.commit()
print("Graph tables created: AUTHORS, PAPER_AUTHORS, PAPER_SIMILARITIES")

## 3.3 Ingest Data into Oracle

Each row is packed into a tuple of `(arxiv_id, title, abstract, text, embedding)`. 

The embedding is converted to a Python `array.array('f', ...)` — Oracle's driver maps this directly to a `VECTOR` column without serialisation overhead.

`executemany` sends the entire batch in a single round trip, which is significantly faster than row-by-row inserts.

In [ ]:
from tqdm import tqdm
import array

rows = []
for i, row in dataset_df.iterrows():
    embedding_array = array.array('f', row.get("embedding"))
    rows.append((
        row.get("arxiv_id"),
        row.get("title"),
        row.get("abstract"),
        row.get("text"),
        embedding_array
    ))

print(f"Preparing to insert {len(rows)} rows into RESEARCH_PAPERS...")

with conn.cursor() as cur:
    for row in tqdm(rows):
        cur.execute("""
            INSERT INTO research_papers (arxiv_id, title, abstract, text, embedding)
            VALUES (:1, :2, :3, :4, :5)
        """, row)
conn.commit()
print("Data inserted successfully")

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM RESEARCH_PAPERS")
    print("Row count:", cur.fetchone()[0])

    cur.execute("""
        SELECT arxiv_id, title, abstract, text FROM RESEARCH_PAPERS
        FETCH FIRST 3 ROWS ONLY
    """)
    for row in cur.fetchall():
        print(row)

## 3.4 Build and Register Graph Relationships

### 3.4.1 Load Author Edges (`Author -> WROTE -> Paper`)

We normalise the raw author strings into a consistent list, then insert each unique author and each author-paper relationship into relational tables. 
These rows become the `WROTE` edges in the property graph — connecting authors to the papers they wrote.

In [ ]:
def normalize_authors(raw_authors):
    if raw_authors is None:
        return []
    if isinstance(raw_authors, str):
        return [a.strip() for a in raw_authors.split(",") if a.strip()]
    if isinstance(raw_authors, list):
        return [str(a).strip() for a in raw_authors if str(a).strip()]
    return []

author_rows = set()
paper_author_rows = set()

for _, row in dataset_df.iterrows():
    arxiv_id = row.get("arxiv_id")
    if not arxiv_id:
        continue
    for author_name in normalize_authors(row.get("authors")):
        author_rows.add((author_name,))
        paper_author_rows.add((arxiv_id, author_name))

with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE paper_authors")
    cur.execute("TRUNCATE TABLE authors")
    if author_rows:
        cur.executemany("INSERT INTO authors (author_name) VALUES (:1)", sorted(author_rows))
    if paper_author_rows:
        cur.executemany("INSERT INTO paper_authors (arxiv_id, author_name) VALUES (:1, :2)", sorted(paper_author_rows))
conn.commit()
print(f"Loaded {len(author_rows)} authors and {len(paper_author_rows)} Author->WROTE->Paper edges.")

### 3.4.2 Load Similarity Edges (`Paper -> SIMILAR_TO -> Paper`)

For each paper, we compute cosine similarity against all other papers using the embedding matrix, then keep the top-K most similar neighbours. 
These become `SIMILAR_TO` edges in the property graph — enabling the graph retrieval strategy to hop from a seed paper to its neighbours.

In [ ]:
TOP_SIM_NEIGHBORS = 10

paper_ids = dataset_df["arxiv_id"].tolist()
emb_matrix = np.vstack(dataset_df["embedding"].values).astype(np.float32)

neighbor_k = min(TOP_SIM_NEIGHBORS, max(len(paper_ids) - 1, 0))
similarity_rows = []

if neighbor_k > 0:
    similarity_matrix = emb_matrix @ emb_matrix.T
    np.fill_diagonal(similarity_matrix, -np.inf)

    for i, source_arxiv_id in enumerate(paper_ids):
        nn_idx = np.argpartition(similarity_matrix[i], -neighbor_k)[-neighbor_k:]
        nn_idx = nn_idx[np.argsort(similarity_matrix[i][nn_idx])[::-1]]
        for rank_no, j in enumerate(nn_idx, start=1):
            similarity_rows.append((
                source_arxiv_id, paper_ids[j],
                float(similarity_matrix[i][j]), rank_no
            ))

with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE paper_similarities")
    if similarity_rows:
        cur.executemany("""
            INSERT INTO paper_similarities (source_arxiv_id, target_arxiv_id, sim_score, rank_no)
            VALUES (:1, :2, :3, :4)
        """, similarity_rows)
conn.commit()
print(f"Loaded {len(similarity_rows)} Paper->SIMILAR_TO->Paper edges (top {neighbor_k} per paper).")

### 3.4.3 Register the Oracle SQL Property Graph

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*) FROM user_property_graphs WHERE graph_name = 'RESEARCH_GRAPH'
    """)
    if cur.fetchone()[0] > 0:
        cur.execute("DROP PROPERTY GRAPH research_graph")

    cur.execute("""
        CREATE PROPERTY GRAPH research_graph
        VERTEX TABLES (
            research_papers KEY (arxiv_id) LABEL paper
                PROPERTIES (arxiv_id, title, abstract),
            authors KEY (author_name) LABEL author
                PROPERTIES (author_name)
        )
        EDGE TABLES (
            paper_authors KEY (arxiv_id, author_name)
                SOURCE KEY (author_name) REFERENCES authors (author_name)
                DESTINATION KEY (arxiv_id) REFERENCES research_papers (arxiv_id)
                LABEL wrote,
            paper_similarities KEY (source_arxiv_id, target_arxiv_id)
                SOURCE KEY (source_arxiv_id) REFERENCES research_papers (arxiv_id)
                DESTINATION KEY (target_arxiv_id) REFERENCES research_papers (arxiv_id)
                LABEL similar_to
                PROPERTIES (sim_score, rank_no)
        )
    """)
conn.commit()
print("SQL Property Graph RESEARCH_GRAPH created.")

> 💡 **Takeaway — Schema Design Drives Retrieval Quality:** The table you built isn't just storage — it's your retrieval foundation. The `VECTOR` column enables semantic search, the Oracle Text index enables keyword search, and the graph tables enable relationship discovery. Each index you created in this section unlocks a different retrieval strategy in Part 4. Without the right schema, even the best retrieval algorithm has nothing to work with.

# Part 4: Retrieval Mechanisms

![Information Retrieval](../images/IR.png)

> 📖 **Companion guide:** See [docs/part-4-retrieval.md](../docs/part-4-retrieval.md)

> 💡 **Key Insight:** No single retrieval strategy wins everywhere. Keyword search excels at exact matches, vector search captures semantic similarity, 
hybrid combines both, and graph retrieval surfaces structural relationships. The best RAG systems let you choose per query.

In [ ]:
SEARCH_TEXT_KEYWORDS = "optimization"

## 4.1 Text Based Retrieval

![Lexical Retrieval](../images/lexical_retrieval.png)

Keyword search uses Oracle Text's full-text index to find papers containing specific terms.

In [ ]:
def keyword_search_research_papers(conn, keyword: str):
    """Perform a full-text keyword search using the Oracle Text index."""
    query = """
        SELECT arxiv_id, title, SUBSTR(text, 1, 200) AS text_snippet,
               SCORE(1) AS relevance_score
        FROM research_papers
        WHERE CONTAINS(text, :keyword, 1) > 0
        ORDER BY SCORE(1) DESC
        FETCH FIRST 10 ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(query, keyword=keyword)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
    return rows, columns

In [ ]:
rows, columns = keyword_search_research_papers(conn, SEARCH_TEXT_KEYWORDS)
results_df = pd.DataFrame(rows, columns=columns)
results_df

## 4.2 Vector Based Retrieval

![Vector Retrieval](../images/vector_retrieval.png)

Vector search finds papers by semantic similarity — matching meaning rather than exact keywords.

In [ ]:
import array

def vector_search_research_papers(conn, embedding_model, search_query: str, top_k: int = 5):
    """Perform a vector similarity search on the research_papers table."""
    query_embedding = embedding_model.encode(
        [f"search_query: {search_query}"],
        convert_to_numpy=True, normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    query = f"""
        SELECT arxiv_id, title, abstract,
               SUBSTR(text, 1, 200) AS text_snippet,
               ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
        FROM research_papers
        ORDER BY similarity_score DESC
        FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
    """
    with conn.cursor() as cur:
        cur.execute(query, q=query_embedding_array)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
    return rows, columns

In [ ]:
rows, columns = vector_search_research_papers(conn, embedding_model, SEARCH_TEXT_KEYWORDS, top_k=5)
results_df = pd.DataFrame(rows, columns=columns)
results_df

> 💡 **Takeaway — Approximate Is Fast Enough:** `FETCH APPROX FIRST k ROWS ONLY WITH TARGET ACCURACY 90` is the key line. It tells Oracle to use the HNSW index instead of scanning every vector. You trade a small amount of recall (guaranteed 90%+) for orders-of-magnitude speed improvement. In production with millions of vectors, this is the difference between milliseconds and seconds.

## 4.3 Hybrid Retrieval (Vector + Text Combined)

Hybrid retrieval combines keyword matching (Oracle Text) with vector similarity in a single query. 
This captures both exact term matches and semantic relevance — covering cases where either strategy alone would miss results.

Oracle supports two filtering strategies (pre-filter and post-filter) plus a fusion approach (RRF). 

Oracle also supports **in-filter** (inline filtering within the vector index scan), though we focus on pre/post-filter and RRF in this workshop.

### 4.3.1 Pre-Filtering

**Pre-filtering** applies the keyword filter *before* vector ranking. Oracle Text narrows the candidate set first, 
then vector distance ranks only the surviving rows. This is fast but may miss semantically relevant papers that don't contain the exact keyword.

In [ ]:
def hybrid_search_research_papers_pre_filter(
    conn, embedding_model, search_phrase: str,
    top_k: int = 10, show_explain: bool = False
):
    """Hybrid search: keyword pre-filter + vector ranking."""
    query_embedding = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True, normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    with conn.cursor() as cur:
        sql = f"""
            SELECT arxiv_id, title, abstract,
                   SUBSTR(text, 1, 200) AS text_snippet,
                   ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
            FROM research_papers
            WHERE CONTAINS(text, :kw, 1) > 0
            ORDER BY similarity_score DESC
            FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
        """
        cur.execute(sql, q=query_embedding_array, kw=search_phrase)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
    return rows, columns, None

In [ ]:
rows, columns, _ = hybrid_search_research_papers_pre_filter(
    conn, embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, show_explain=False
)
pre_filter_results_df = pd.DataFrame(rows, columns=columns)
pre_filter_results_df

### 4.3.2 Post-Filtering

**Post-filtering** reverses the order — vector search runs first to find the top candidates by semantic similarity, 
then a keyword filter removes results that don't match the text criteria. This gives broader recall but may be slower on large tables.

In [ ]:
def hybrid_search_research_papers_postfilter(
    conn, embedding_model, search_phrase: str,
    top_k: int = 10, candidate_k: int = 200, show_explain: bool = False
):
    """Hybrid search: vector candidates first, then text filter."""
    query_embedding = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True, normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    with conn.cursor() as cur:
        sql = f"""
            WITH vec_candidates AS (
                SELECT arxiv_id, title, abstract, text,
                       1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS similarity_score
                FROM research_papers
                ORDER BY similarity_score DESC
                FETCH APPROX FIRST {candidate_k} ROWS ONLY WITH TARGET ACCURACY 90
            )
            SELECT arxiv_id, title,
                   SUBSTR(text, 1, 200) AS text_snippet,
                   ROUND(similarity_score, 4) AS similarity_score
            FROM vec_candidates
            WHERE CONTAINS(text, :kw, 1) > 0
            ORDER BY similarity_score DESC
            FETCH FIRST {top_k} ROWS ONLY
        """
        cur.execute(sql, q=query_embedding_array, kw=search_phrase)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
    return rows, columns, None

In [ ]:
rows, columns, _ = hybrid_search_research_papers_postfilter(
    conn, embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, candidate_k=200, show_explain=False
)
post_filter_results_df = pd.DataFrame(rows, columns=columns)
post_filter_results_df

| Approach | Strength | Best For |
|---|---|---|
| Pre-filter | Fast; reduces candidate set early | Known keywords + semantic ranking |
| Post-filter | Broader semantic recall first | Exploratory queries with keyword refinement |

### 4.3.3 Reciprocal Rank Fusion

**Reciprocal Rank Fusion (RRF)** merges results from two independent ranked lists — one from vector search, one from keyword search. 
Each result's score is based on its *rank position* (not raw score), using the formula `1 / (k + rank)`. 
This normalises across scoring scales and gives a balanced blend of both retrieval signals.

RRF is simple, effective, and doesn't require tuning weights between the two strategies.

In [ ]:
def hybrid_rrf_search(
    conn, embedding_model, search_phrase: str,
    top_k: int = 10, per_list: int = 120, k: int = 60,
    phrase_safe: bool = True, show_explain: bool = False
):
    """RRF fusion of Vector + Oracle Text results."""
    qv = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True, normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    qv = array.array('f', qv)
    kw = f"\"{search_phrase}\"" if (phrase_safe and " " in search_phrase.strip()) else search_phrase

    with conn.cursor() as cur:
        sql = f"""
            WITH
            vec AS (
              SELECT arxiv_id, title, SUBSTR(text, 1, 200) AS text_snippet,
                     1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS sim_vec,
                     ROW_NUMBER() OVER (ORDER BY 1 - VECTOR_DISTANCE(embedding, :q, COSINE) DESC) AS r_vec
              FROM research_papers
              FETCH APPROX FIRST {per_list} ROWS ONLY WITH TARGET ACCURACY 90
            ),
            txt AS (
              SELECT arxiv_id, title, SUBSTR(text, 1, 200) AS text_snippet,
                     SCORE(1) AS score_txt,
                     ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
              FROM research_papers
              WHERE CONTAINS(text, :kw, 1) > 0
              FETCH FIRST {per_list} ROWS ONLY
            ),
            fused AS (
              SELECT COALESCE(v.arxiv_id, t.arxiv_id) AS arxiv_id,
                     COALESCE(v.title, t.title) AS title,
                     COALESCE(v.text_snippet, t.text_snippet) AS text_snippet,
                     NVL(v.r_vec, 999999) AS r_vec,
                     NVL(t.r_txt, 999999) AS r_txt,
                     NVL(v.sim_vec, 0) AS sim_vec,
                     NVL(t.score_txt, 0) AS score_txt
              FROM vec v FULL OUTER JOIN txt t ON t.arxiv_id = v.arxiv_id
            )
            SELECT arxiv_id, title, text_snippet,
                   ROUND((1.0/(:k + r_vec)) + (1.0/(:k + r_txt)), 6) AS rrf_score,
                   r_vec, r_txt, ROUND(sim_vec, 4) AS sim_vec, ROUND(score_txt, 4) AS score_txt
            FROM fused
            ORDER BY rrf_score DESC, title
            FETCH FIRST {top_k} ROWS ONLY
        """
        cur.execute(sql, q=qv, kw=kw, k=k)
        rows = cur.fetchall()
        columns = [d[0] for d in cur.description]
    return rows, columns, None

In [ ]:
rows, columns, _ = hybrid_rrf_search(
    conn, embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS,
    top_k=5, per_list=60, k=60, show_explain=False
)
rrf_results_df = pd.DataFrame(rows, columns=columns)
rrf_results_df

> 💡 **Takeaway — No Single Strategy Wins:** Pre-filter is precise when you know a keyword must appear. Post-filter gives broad semantic recall with keyword validation. RRF fuses both signals without tuning weights. The right choice depends on the query — and a production system often needs all three available.

## 4.4 Graph-Based Retrieval (Oracle SQL Property Graph)

Graph retrieval combines vector similarity with structural relationships. The strategy:

1. **Seed** — find the top-K papers by vector distance (same as vector search)
2. **Expand** — traverse `SIMILAR_TO` and `WROTE` edges to discover related papers the vector search might miss
3. **Score** — blend vector similarity with graph proximity into a final ranking

This surfaces papers that are structurally connected (same author, similar topic cluster) even if their embeddings aren't the closest match.

![Graph Retrieval](../images/graph_retrieval.png)

In [ ]:
def graph_search_research_papers(
    conn, embedding_model, search_query: str, top_k: int = 10, seed_k: int = 25
):
    """Graph retrieval using Oracle SQL Property Graph + GRAPH_TABLE."""
    seed_k = max(seed_k, top_k)
    query_embedding = embedding_model.encode(
        [f"search_query: {search_query}"],
        convert_to_numpy=True, normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    sql = f"""
        WITH seed AS (
            SELECT arxiv_id, 1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS seed_score
            FROM research_papers
            ORDER BY seed_score DESC
            FETCH APPROX FIRST {seed_k} ROWS ONLY WITH TARGET ACCURACY 90
        ),
        seed_hits AS (
            SELECT arxiv_id AS source_arxiv_id, arxiv_id AS candidate_arxiv_id,
                   seed_score, 'seed' AS relation_type, seed_score AS edge_score
            FROM seed
        ),
        sim_hops AS (
            SELECT s.arxiv_id AS source_arxiv_id, gt.target_arxiv_id AS candidate_arxiv_id,
                   s.seed_score, 'similar_to' AS relation_type, gt.edge_score AS edge_score
            FROM seed s
            JOIN GRAPH_TABLE(
                research_graph
                MATCH (src IS paper)-[e IS similar_to]->(dst IS paper)
                COLUMNS (src.arxiv_id AS source_arxiv_id, dst.arxiv_id AS target_arxiv_id, e.sim_score AS edge_score)
            ) gt ON gt.source_arxiv_id = s.arxiv_id
        ),
        author_hops AS (
            SELECT s.arxiv_id AS source_arxiv_id, gt.target_arxiv_id AS candidate_arxiv_id,
                   s.seed_score, 'shared_author' AS relation_type, 1.0 AS edge_score
            FROM seed s
            JOIN GRAPH_TABLE(
                research_graph
                MATCH (src IS paper)<-[w1 IS wrote]-(a IS author)-[w2 IS wrote]->(dst IS paper)
                COLUMNS (src.arxiv_id AS source_arxiv_id, dst.arxiv_id AS target_arxiv_id)
            ) gt ON gt.source_arxiv_id = s.arxiv_id
            WHERE gt.target_arxiv_id <> s.arxiv_id
        ),
        candidates AS (
            SELECT * FROM seed_hits UNION ALL
            SELECT * FROM sim_hops UNION ALL
            SELECT * FROM author_hops
        ),
        scored AS (
            SELECT candidate_arxiv_id AS arxiv_id,
                   MAX(CASE relation_type
                       WHEN 'seed' THEN seed_score
                       WHEN 'similar_to' THEN (0.70 * seed_score) + (0.30 * edge_score)
                       WHEN 'shared_author' THEN (0.85 * seed_score) + (0.15 * edge_score)
                       ELSE seed_score END) AS graph_score
            FROM candidates GROUP BY candidate_arxiv_id
        )
        SELECT rp.arxiv_id, rp.title, rp.abstract,
               SUBSTR(rp.text, 1, 200) AS text_snippet,
               ROUND(sc.graph_score, 4) AS graph_score
        FROM scored sc JOIN research_papers rp ON rp.arxiv_id = sc.arxiv_id
        ORDER BY graph_score DESC
        FETCH FIRST {top_k} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, q=query_embedding_array)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
    return rows, columns

In [ ]:
rows, columns = graph_search_research_papers(
    conn, embedding_model, search_query=SEARCH_TEXT_KEYWORDS, top_k=5, seed_k=20
)
graph_results_df = pd.DataFrame(rows, columns=columns)
graph_results_df

## 4.5 Compare Top Results Across Retrieval Strategies

In [ ]:
def extract_top_titles(rows, columns, k=5):
    """Return top-k titles from retrieval results."""
    titles = []
    for row in rows[:k]:
        row_data = dict(zip(columns, row))
        titles.append(row_data.get("TITLE", "Untitled"))
    return titles

comparison_specs = [
    ("keyword", lambda: keyword_search_research_papers(conn, SEARCH_TEXT_KEYWORDS)),
    ("vector", lambda: vector_search_research_papers(conn, embedding_model, SEARCH_TEXT_KEYWORDS, top_k=5)),
    ("hybrid_pre_filter", lambda: hybrid_search_research_papers_pre_filter(
        conn=conn, embedding_model=embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, show_explain=False)),
    ("hybrid_postfilter", lambda: hybrid_search_research_papers_postfilter(
        conn=conn, embedding_model=embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, candidate_k=200, show_explain=False)),
    ("hybrid_rrf", lambda: hybrid_rrf_search(
        conn=conn, embedding_model=embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, per_list=60, k=60, show_explain=False)),
    ("graph", lambda: graph_search_research_papers(
        conn=conn, embedding_model=embedding_model, search_query=SEARCH_TEXT_KEYWORDS, top_k=5, seed_k=20)),
]

comparison_rows = []
for strategy_name, runner in comparison_specs:
    result = runner()
    if isinstance(result, tuple) and len(result) >= 2:
        rows, columns = result[0], result[1]
    else:
        rows, columns = [], []
    titles = extract_top_titles(rows, columns, k=5)
    record = {"retrieval_strategy": strategy_name}
    for i in range(5):
        record[f"Top_{i+1}"] = titles[i] if i < len(titles) else ""
    comparison_rows.append(record)

retrieval_strategy_comparison_df = pd.DataFrame(comparison_rows)
retrieval_strategy_comparison_df

> 💡 **Takeaway — Retrieval Is the Hard Part:** The comparison table shows that the same query can return very different papers depending on the strategy. Graph retrieval surfaces connections that keyword and vector search miss entirely. This is why strategy selection directly impacts RAG answer quality — a great LLM cannot compensate for poor retrieval.

# Part 5: Building a RAG Pipeline

> 📖 **Companion guide:** See [docs/part-5-rag-pipeline.md](../docs/part-5-rag-pipeline.md) Flow:

1. Configure API access and initialize the OpenAI client
2. Define a reusable RAG function supporting multiple retrieval modes
3. Run an end-to-end query

> 💡 **Key Term — RAG (Retrieval-Augmented Generation):** A pattern where the LLM receives retrieved context alongside the user query. 
The model generates answers grounded in real data rather than relying solely on its training knowledge.

## 5.1 Configure API Access

In [ ]:
import os

oci_genai_api_key = os.environ.get("OCI_GENAI_API_KEY")
assert oci_genai_api_key, "OCI_GENAI_API_KEY not set — check your Codespace environment variables"
print("OCI GenAI API key loaded")


## 5.2 Initialize and Smoke-Test the OpenAI Client

In [ ]:
from openai import OpenAI

OCI_GENAI_ENDPOINT = os.environ.get(
    "OCI_GENAI_ENDPOINT",
    "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com/openai/v1"
)
OPENAI_MODEL = "xai.grok-3-fast"

openai_client = OpenAI(base_url=OCI_GENAI_ENDPOINT, api_key=oci_genai_api_key)

response = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": "You are a research paper assistant."},
        {"role": "user", "content": "Hello! I'm a user!"},
    ],
)
print(response.choices[0].message.content)


## 5.3 Define the Reusable RAG Function

In [ ]:
def research_paper_assistant_rag_pipeline(
    conn, embedding_model, user_query: str,
    top_k: int = 10, retrieval_mode: str = "hybrid", show_explain: bool = False
):
    """RAG pipeline: retrieve from Oracle, format context, call LLM."""
    # 1. Retrieve
    if retrieval_mode == "keyword":
        rows, columns = keyword_search_research_papers(conn, user_query)
    elif retrieval_mode == "vector":
        rows, columns = vector_search_research_papers(conn, embedding_model, user_query, top_k)
    elif retrieval_mode == "graph":
        rows, columns = graph_search_research_papers(conn, embedding_model, user_query, top_k=top_k)
    else:
        rows, columns, _ = hybrid_search_research_papers_pre_filter(
            conn=conn, embedding_model=embedding_model,
            search_phrase=user_query, top_k=top_k, show_explain=show_explain
        )

    retrieved_count = len(rows) if rows else 0
    print(f"Retrieved {retrieved_count} papers using {retrieval_mode.upper()} retrieval.")

    # 2. Format context
    formatted_context = ""
    if retrieved_count > 0:
        formatted_context += f"\n\n{retrieved_count} relevant research papers retrieved:\n\n"
        for i, row in enumerate(rows):
            row_data = dict(zip(columns, row))
            title = row_data.get("TITLE", "Untitled Paper")
            abstract = row_data.get("ABSTRACT", "No abstract available.")
            snippet = row_data.get("TEXT_SNIPPET", "")
            score = (row_data.get("GRAPH_SCORE") or row_data.get("SIMILARITY_SCORE")
                     or row_data.get("RELEVANCE_SCORE") or "N/A")
            formatted_context += (
                f"[{i+1}] **{title}**\n"
                f"Abstract: {abstract}\nSnippet: {snippet}\nRelevance Score: {score}\n\n"
            )
    else:
        formatted_context = "\n\nNo relevant papers were retrieved.\n"

    # 3. Call LLM
    prompt = f"""You are a Research Paper Assistant.
User Query: {user_query}
Retrieved papers: {retrieved_count}
{formatted_context}
Summarize findings. Use [X] citations. Highlight consensus and gaps."""

    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a scientific research assistant. Use only provided context. Cite papers [1], [2], etc."},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content

## 5.4 Run an End-to-End RAG Example

![Full RAG Pipeline](../images/full_rag_pipeline.png)

> **Be patient:** This cell sends the retrieved context to the LLM for summarization. Expect it to take around **30–60 seconds** depending on the model and network latency.

In [ ]:
summary = research_paper_assistant_rag_pipeline(
    conn=conn, embedding_model=embedding_model,
    user_query=SEARCH_TEXT_KEYWORDS, top_k=10,
    retrieval_mode="hybrid", show_explain=False
)
print(summary)

> 💡 **Takeaway — The Full RAG Loop:** You just completed the retrieve-then-generate pattern end to end: user question → Oracle retrieval → formatted context with citations → LLM generation → grounded answer. Every piece matters: the embedding quality from Part 2, the schema from Part 3, the retrieval strategy from Part 4, and the prompt design from this section. Change any one and the output quality shifts.

# Where to Next?

You've built a complete information retrieval and RAG pipeline. Here are some paths to continue:

- **[From RAG to Agents Workshop](https://github.com/speechlyze/from_rag_to_agents_workshop)** — Continue the journey by adding AI agents, multi-agent orchestration, and persistent session memory on top of this RAG pipeline
- **[Oracle AI Developer Hub](https://github.com/oracle-devrel/oracle-ai-developer-hub)** — More technical assets, samples, and projects with Oracle AI
- **[Oracle Developer Resource](https://www.oracle.com/developer/)** — Documentation, tools, and community for Oracle developers